# Exp 14: Attention dropping

Card: [exp14_attention_dropping.md](../../experiments/exp14_attention_dropping.md)

In [ ]:
import dataclasses
import os

import albumentations as A
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from PIL import Image

from core.config import CLASSES, IMG_SIZE, MLFLOW_URI, ROOT, SEED, TRAIN_DIR
from core.cv import VIP, Config, CrossValidator, CVResult
from core.data import attention_drop_tfms, read_boxes, read_train
from core.gradcam import GradCamFigures
from core.predict import best_f1

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-attention-dropping-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SHOW = 2  # images per exp02 Grad-CAM figure
    N_POSITIVES = 2  # images per exp12 Grad-CAM figure
else:
    OVERRIDES = {}
    N_SHOW = 4
    N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp14_attention_dropping" / "resnet18.toml", **OVERRIDES
)
# trained in exp11 on folds 0-4, only read; compared on same folds as CONFIG
EXP11 = dataclasses.replace(
    Config.from_toml(CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.toml"),
    folds=CONFIG.folds,
)
# case images of docs/gradcam_augmentations.md, VIP cases
CASES = {
    "beside cat": 779,
    "on cat": 9101,
    "on rug": 2470,
}
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp14_attention_dropping"
SAVE_FIGURES = not SMOKE
PROBS = [f"p_{c}" for c in CLASSES]

# dataviz reference palette, light mode; default keeps exp06 baseline color
MUTED = "#898781"
SERIES = {"default (exp11)": "#eb6834", "attention_drop": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()

## attention_drop draws

In [ ]:
N_DRAWS = 6
train_df = read_train()
only_cat = train_df[CLASSES].eq([1, 0, 0, 0, 0]).all(axis=1)
picks = {cls: train_df[train_df[cls] == 1] for cls in VIP} | {
    "Cat only": train_df[only_cat]
}
image_ids = {
    name: rows.sample(1, random_state=SEED)["image_id"].item()
    for name, rows in picks.items()
}
images = {
    name: np.array(
        Image.open(TRAIN_DIR / f"{image_id}.jpg")
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )
    for name, image_id in image_ids.items()
}
boxes = read_boxes()
BOX = SERIES["attention_drop"]
tfms = attention_drop_tfms(IMG_SIZE)
show = A.Compose(
    tfms.transforms[:-2], bbox_params=tfms.processors["bboxes"].params
)  # without Normalize, ToTensorV2


def draw(name, original=False):
    """(image, boxes, box labels): original, or one attention_drop draw. Boxes follow every transform."""
    image_boxes = boxes.get(image_ids[name], [])
    bboxes = np.array([b for _, b in image_boxes]).reshape(-1, 4)
    labels = [c for c, _ in image_boxes]
    if original:
        return images[name], bboxes, labels
    out = show(image=images[name], bboxes=bboxes, box_labels=labels)
    return out["image"], out["bboxes"], out["box_labels"]


def plot_boxes(ax, image, bboxes, labels):
    h, w = image.shape[:2]
    for (x0, y0, x1, y1), cls in zip(bboxes, labels):
        # dashed: area > 0.3 of image, never erased
        ls = "--" if (x1 - x0) * (y1 - y0) > tfms.transforms[0].max_area else "-"
        ax.add_patch(
            plt.Rectangle(
                (x0 * w, y0 * h),
                (x1 - x0) * w,
                (y1 - y0) * h,
                fill=False,
                ec=BOX,
                lw=1.5,
                ls=ls,
            )
        )
        ax.text(
            x0 * w + 2,
            y0 * h + 2,
            cls,
            color="white",
            fontsize=7,
            va="top",
            bbox=dict(fc=BOX, ec="none", pad=1),
        )


fig, axes = plt.subplots(
    len(images), N_DRAWS + 1, figsize=(2.1 * (N_DRAWS + 1), 2.4 * len(images))
)
for name, row_axes in zip(images, axes):
    for i, ax in enumerate(row_axes):
        shown, bboxes, labels = draw(name, original=i == 0)
        ax.imshow(shown)
        plot_boxes(ax, shown, bboxes, labels)
        ax.set_axis_off()
        if name == VIP[0]:
            ax.set_title("Original" if i == 0 else f"Draw {i}", fontsize=9)
    row_axes[0].text(
        -0.08,
        0.5,
        name,
        transform=row_axes[0].transAxes,
        rotation=90,
        va="center",
        ha="right",
    )
fig.tight_layout()
figures = {f"{PREFIX}_draws.png": fig}
plt.show()

## Training

In [ ]:
results = {"default (exp11)": CVResult.load(EXP11)}
# exp11 result holds folds 0-4: keep folds of this experiment
base = results["default (exp11)"]
results["default (exp11)"] = dataclasses.replace(
    base,
    config=EXP11,
    oof=base.oof[base.oof["fold"].isin(CONFIG.folds)],
    fold_ap=base.fold_ap.loc[CONFIG.folds],
    run_ids={fold: base.run_ids[fold] for fold in CONFIG.folds},
)

In [ ]:
results["attention_drop"] = CrossValidator(CONFIG).run()

## Training curves

In [ ]:
def mean_curve(result, key):
    # epoch-level metrics come one per epoch, in step order
    runs = [
        client.get_metric_history(run_id, key) for run_id in result.run_ids.values()
    ]
    return np.mean(
        [[m.value for m in sorted(run, key=lambda m: m.step)] for run in runs], axis=0
    )


panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, mean_curve(r, train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, mean_curve(r, val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(CONFIG.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="augmentations", alignment="left")
fig.tight_layout()
figures[f"{PREFIX}_training.png"] = fig
plt.show()

## Scores

In [ ]:
def fold_scores(result):
    """AP per class, mAP and Macro F1 (best-F1 thresholds tuned on same fold rows), one row per fold."""
    macro_f1 = {
        fold: best_f1(part[CLASSES].to_numpy(), part[PROBS].to_numpy())["f1"].mean()
        for fold, part in result.oof.groupby("fold")
    }
    return result.fold_ap.assign(**{"Macro F1": pd.Series(macro_f1)})


scores = pd.concat(
    {name: fold_scores(r) for name, r in results.items()}, names=["augmentations"]
)
mean = scores.groupby("augmentations", sort=False).mean()
pd.concat([scores, mean.set_index(mean.index + " mean")]).round(3)

In [ ]:
# difference to exp11 counts only if it has same sign on both folds
diff = (
    (scores.loc["attention_drop"] - scores.loc["default (exp11)"])
    .T.add_prefix("diff fold ")
)
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Grad-CAM: case images

In [ ]:
cams = {name: GradCamFigures(r.config) for name, r in results.items()}
case_classes = [
    (case, idx, cls)
    for case, idx in CASES.items()
    for cls in VIP
    if train_df.loc[idx, cls] == 1
]
columns = [(name, fold) for name in cams for fold in CONFIG.folds]
fig, axes = plt.subplots(
    len(case_classes),
    len(columns) + 1,
    figsize=(2.6 * (len(columns) + 1), 3 * len(case_classes)),
    squeeze=False,
)
for (case, idx, cls), row_axes in zip(case_classes, axes):
    for k, (name, fold) in enumerate(columns):
        shown, cam, prob = cams[name].image_and_cam(idx, cls, fold)
        if k == 0:
            row_axes[0].imshow(shown)
            row_axes[0].set_title(f"{case} · {cls}", fontsize=9)
        row_axes[k + 1].imshow(shown)
        row_axes[k + 1].imshow(cam, cmap="jet", vmin=0, vmax=1, alpha=0.5)
        role = "held out" if train_df.loc[idx, "fold"] == fold else "trained"
        row_axes[k + 1].set_title(
            f"{name}\nfold {fold} ({role}) · p {prob:.3f}",
            fontsize=8,
            color=SERIES[name],
        )
for ax in axes.flat:
    ax.set_axis_off()
fig.tight_layout()
figures[f"{PREFIX}_grad_cam_cases.png"] = fig
plt.show()

## Grad-CAM: correct and incorrect (exp02)

In [ ]:
new_cams = cams["attention_drop"]
for (cls, outcome), fig in new_cams.figures(VIP, N_SHOW).items():
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

## Grad-CAM: validation and train (exp12)

In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = new_cams.positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

In [ ]:
with mlflow.start_run(
    experiment_id=client.get_experiment_by_name(CONFIG.experiment).experiment_id,
    run_name="summary",
):
    mlflow.log_param("val_mode", "fast")
    mlflow.log_table(scores.reset_index(), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_metrics({f"threshold_{c}": new_cams.thresholds[c] for c in VIP})
    if SAVE_FIGURES:
        for fname, fig in figures.items():
            fig.savefig(FIGURES / fname, dpi=120)
            mlflow.log_figure(fig, fname)